# ArchNetv2-openings — Kaggle background training (auto-resume)

Trains the 4-head + AC-CBAM ArchNetv2 (door + window) on CubiCasa5K.
Designed for Kaggle's **Save & Run All** detached batch jobs: each run
trains for <12 h, then checkpoints to a Kaggle Dataset. Re-run as many
times as needed — it resumes automatically until it hits 500 epochs or
early-stops.

## One-time setup (do this before the first run)
1. **Settings → Accelerator → GPU T4 x2** (or P100).
2. **Settings → Internet → On**.
3. **Add data**: search Kaggle Datasets for `cubicasa5k` and add it.
   Note the mount path (usually `/kaggle/input/cubicasa5k`).
4. Create an **empty checkpoint dataset** once so resume has a target:
   `Create → New Dataset` → upload any tiny placeholder file → name it
   e.g. `archnetv2-ckpt`. Its slug is `<your-username>/archnetv2-ckpt`.
5. **Add-ons → Secrets**: add `KAGGLE_USERNAME` and `KAGGLE_KEY` (from
   your kaggle.json) so the notebook can push checkpoints via the API.

Then just keep clicking **Save Version → Save & Run All (Commit)**.

## 1. Environment + Kaggle API creds

In [ ]:
!pip install -q ultralytics==8.4.58
import os
# Wire Kaggle API creds from Notebook Secrets (for checkpoint upload).
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = s.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = s.get_secret('KAGGLE_KEY')
    print('Kaggle API creds loaded from Secrets.')
except Exception as e:
    print('No Secrets configured — checkpoint upload will be skipped:', e)
!nvidia-smi | head -15

## 2. Get the code
Clone this public repo (or upload it as a Dataset and adjust the path).

In [ ]:
import os, sys
REPO = '/kaggle/working/training-archnetv2'
if os.path.isdir(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/miladmirzazadeh/training-archnetv2.git {REPO}
os.chdir(REPO)
sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

## 3. Locate CubiCasa5K and convert to YOLO labels (symlinked)
Symlinks keep `/kaggle/working` tiny — the 9 GB of source PNGs stay in
`/kaggle/input`. Skips conversion if labels already exist.

In [ ]:
import glob, os
# Auto-detect the CubiCasa5K mount (folder containing train.txt).
cands = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train.txt', recursive=True)]
assert cands, 'CubiCasa5K not found in /kaggle/input — add it via "Add data".'
CUBI = cands[0]
print('CubiCasa5K root:', CUBI)

YOLO_DS = '/kaggle/working/cubicasa5k_yolo'
# Re-convert unless training images already exist (guards against a prior
# failed run that wrote an empty data.yaml).
n_train = len(glob.glob(f'{YOLO_DS}/images/train/*'))
if n_train == 0:
    !python -m archnetv2.scripts.prepare_cubicasa5k \
        --src {CUBI} --dst {YOLO_DS} --quality all --link-images
else:
    print(f'YOLO dataset already prepared ({n_train} train images).')

## 4. Train with auto-resume + time budget
Set `CKPT` to your checkpoint dataset slug. The first run starts fresh;
every subsequent **Save & Run All** resumes from the pushed checkpoint.

In [ ]:
CKPT = 'YOUR_USERNAME/archnetv2-ckpt'  # <-- edit this
!python -m archnetv2.scripts.kaggle_resume_train \
    --data {YOLO_DS}/data.yaml \
    --ckpt-dataset {CKPT} \
    --time-budget 11.0 \
    --epochs 500 --batch 4 --imgsz 640 --patience 100 \
    --device 0

## 5. Quick check — best.pt and a sample prediction
After training stops (time budget or convergence), inspect the weights.

In [ ]:
!ls -la /kaggle/working/runs/archnetv2/archnetv2_openings/weights/ 2>/dev/null || echo 'no weights yet'

## How resume works across sessions
- Each session **restores** the run dir from the `archnetv2-ckpt`
  dataset, **resumes** training, trains ~11 h, then **uploads** a new
  version of `archnetv2-ckpt` containing the updated run dir.
- `best.pt` and `last.pt` live in
  `runs/archnetv2/archnetv2_openings/weights/` inside that checkpoint
  dataset — download it from the dataset page when training finishes.
- Total epochs accumulate across sessions until 500 or early-stop
  (patience=100).